
# Wind + battery optimisation for a grid-connected electrolyser

This notebook is designed to be **self-contained** for the optimisation stage:

- It does **not** import `dashboard_config.py`.
- It does **not** import or depend on the existing wind/grid notebooks.
- It does use the existing **Brightway foreground database** for the electrolyser operation activity.
- It pulls grid emissions/generation from the CSV and wind generation from the Renewables.ninja wind API.
- It searches wind-farm capacity and battery capacity combinations to find configurations that keep hydrogen below a target GWP threshold.

The default electrolyser capacity is set to **1 kW** because that is what was requested. For the earlier 1 MW case, change `ELECTROLYSER_CAPACITY_KW = 1000.0` in the dashboard cell.



## 1. Dashboard — edit this cell first

All controls for location, turbine, hub height, time range, foreground activity, optimisation bounds, and threshold logic are here.


In [7]:

# =============================================================================
# DASHBOARD — all user controls live here
# =============================================================================

# --- Brightway / foreground ---------------------------------------------------
PROJECT_NAME = "hydrogen-smr"
FOREGROUND_DB = "hydrogen foreground"

# Foreground operation process to use. These codes match the existing foreground notebook.
ELECTROLYSER_TECH = "AE operation"      # "AE operation", "PEM operation", or "SOEC operation"
ELECTROLYSER_ACTIVITY_CODES = {
    "AE operation":   "ae_op_hermesmann_1kg_h2",
    "PEM operation":  "pem_op_hermesmann_1kg_h2",
    "SOEC operation": "soec_op_hermesmann_1kg_h2",
}

# Requested scale. Keep at 1.0 for 1 kW; use 1000.0 for 1 MW.
ELECTROLYSER_CAPACITY_KW = 1.0

# Operating modes to test. 1.0 = 100% load; 0.10 = 10% load.
LOAD_FRACTIONS_TO_TEST = [1.0, 0.10]

# LCA method selection. Leave LCA_METHOD = None to search by terms.
LCA_METHOD = None
LCA_METHOD_SEARCH_TERMS = ["IPCC", "2021", "GWP", "100"]
LCA_METHOD_INDEX_IF_MULTIPLE = 0

# --- Grid/emissions CSV -------------------------------------------------------
CSV_PATH = "df_fuel_ckan.csv"
CSV_DATETIME_COL = "DATETIME"
CSV_CARBON_INTENSITY_COL = "CARBON_INTENSITY"   # expected as gCO2e/kWh in your CSV
CSV_CARBON_INTENSITY_UNIT = "gCO2e_per_kWh"     # "gCO2e_per_kWh" or "kgCO2e_per_kWh"

# Run/range controls. These are inclusive bounds after parsing the CSV datetime.
RUN_START = "2025-01-01 00:00:00"
RUN_END   = "2025-12-31 23:30:00"

# If the CSV is missing CARBON_INTENSITY, the notebook can estimate grid intensity
# from generation columns using these editable lifecycle factors.
FUEL_LCA_FACTORS_KGCO2E_PER_KWH = {
    "GAS": 0.42,
    "COAL": 0.85,
    "NUCLEAR": 0.012,
    "WIND": 0.015,
    "WIND_EMB": 0.015,
    "HYDRO": 0.008,
    "IMPORTS": 0.35,
    "BIOMASS": 0.10,
    "OTHER": 0.35,
    "SOLAR": 0.045,
    "STORAGE": 0.05,
}
GENERATION_TOTAL_COL = "GENERATION"

# Optional electricity market loss factor, matching your previous notebooks.
APPLY_GRID_LOSSES = True
MARKET_LOSS_SHARE = 0.031642692177
GRID_LIFECYCLE_UPLIFT_KGCO2E_PER_KWH = 0.0  # add extra upstream grid burden if desired

# --- Renewables.ninja wind API controls --------------------------------------
# Location controls.
WIND_LAT = 51.7320
WIND_LON = -0.3711

# Turbine controls.
NINJA_DATASET = "merra2"
NINJA_TURBINE = "Vestas V90 2000"
NINJA_HUB_HEIGHT_M = 80

# The API is called once at a reference capacity and then scaled linearly.
# This avoids calling the API repeatedly for every wind-farm size.
WIND_API_REFERENCE_CAPACITY_KW = 1.0
WIND_TO_GRID_ALIGNMENT = "ffill"  # "ffill" or "linear"

# Token handling: paste token here, or leave blank and use environment variable
# RENEWABLES_NINJA_TOKEN, or be prompted by getpass.
NINJA_API_TOKEN = ""

# --- Wind electricity LCA -----------------------------------------------------
# If WIND_LCA_SCORE_OVERRIDE is None, the notebook searches Brightway using
# WIND_BACKGROUND_QUERY and runs a 1 kWh LCA for the selected wind process.
WIND_LCA_SCORE_OVERRIDE_KGCO2E_PER_KWH = None
WIND_BACKGROUND_QUERY = "electricity production, wind, 1-3MW turbine, onshore GB"
WIND_BACKGROUND_INDEX = 0
BACKGROUND_DATABASES_TO_SEARCH = None  # None = all non-foreground databases

# Allocation choice for wind infrastructure burden:
#   "dedicated_no_export": count all wind generated by the candidate wind farm, including curtailed electricity.
#   "used_only": count only wind electricity used directly or charged into the battery.
WIND_ALLOCATION_MODE = "dedicated_no_export"

# --- Battery model ------------------------------------------------------------
# The battery is charged only from surplus wind and discharged to avoid grid draw.
BATTERY_ROUNDTRIP_EFFICIENCY = 0.90
BATTERY_INITIAL_SOC_FRACTION = 0.50
BATTERY_C_RATE = 1.0  # max charge/discharge power = BATTERY_C_RATE * battery_capacity_kWh

# Capital burden model for battery infrastructure.
INCLUDE_BATTERY_CAPITAL_IN_LCA = True
BATTERY_EMBODIED_KGCO2E_PER_KWH_CAPACITY = 100.0
BATTERY_LIFETIME_YEARS = 10.0

# --- Optimisation controls ----------------------------------------------------
THRESHOLD_KGCO2E_PER_KGH2 = 2.0

# Compliance modes:
#   "annual_average": annual/period-average kgCO2e/kgH2 must be below threshold.
#   "all_slices": every time slice must be below threshold.
#   "slice_share": at least REQUIRED_SLICE_SHARE of time slices must be below threshold.
COMPLIANCE_MODE = "annual_average"
REQUIRED_SLICE_SHARE = 0.90

# Search bounds. Defaults are scaled to a 1 kW electrolyser; increase for 1 MW.
WIND_CAPACITY_MIN_KW = 0.0
WIND_CAPACITY_MAX_KW = 20.0
WIND_CAPACITY_STEP_KW = 0.1

BATTERY_CAPACITY_MIN_KWH = 0.0
BATTERY_CAPACITY_MAX_KWH = 50.0
BATTERY_CAPACITY_STEP_KWH = 0.1

# Selection rule from the feasible set.
#   "min_wind_then_battery" gives the smallest wind farm that works, then smallest battery.
#   "min_battery_then_wind" gives the smallest battery that works, then smallest wind farm.
#   "min_total_capacity" minimizes wind kW + battery kWh as a simple infrastructure proxy.
OPTIMISATION_OBJECTIVE = "min_wind_then_battery"

# Optional cap for printing progress.
PROGRESS_EVERY_N_CANDIDATES = 500

# --- Outputs -----------------------------------------------------------------
OUTPUT_DIR = "wind_battery_optimisation_outputs"
SAVE_OUTPUTS = True

print("Dashboard loaded")
print(f"Electrolyser: {ELECTROLYSER_TECH} at {ELECTROLYSER_CAPACITY_KW} kW")
print(f"Run range: {RUN_START} -> {RUN_END}")
print(f"Wind site: lat={WIND_LAT}, lon={WIND_LON}, turbine={NINJA_TURBINE}, hub height={NINJA_HUB_HEIGHT_M} m")
print(f"Threshold: {THRESHOLD_KGCO2E_PER_KGH2} kgCO2e/kgH2, compliance={COMPLIANCE_MODE}")


Dashboard loaded
Electrolyser: AE operation at 1.0 kW
Run range: 2025-01-01 00:00:00 -> 2025-12-31 23:30:00
Wind site: lat=51.732, lon=-0.3711, turbine=Vestas V90 2000, hub height=80 m
Threshold: 2.0 kgCO2e/kgH2, compliance=annual_average


## 2. Imports and small validation checks

In [8]:

from pathlib import Path
from getpass import getpass
import os
import math
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

assert ELECTROLYSER_TECH in ELECTROLYSER_ACTIVITY_CODES, "Unknown ELECTROLYSER_TECH"
assert COMPLIANCE_MODE in {"annual_average", "all_slices", "slice_share"}
assert OPTIMISATION_OBJECTIVE in {"min_wind_then_battery", "min_battery_then_wind", "min_total_capacity"}
assert WIND_TO_GRID_ALIGNMENT in {"ffill", "linear"}
assert WIND_ALLOCATION_MODE in {"dedicated_no_export", "used_only"}
assert ELECTROLYSER_CAPACITY_KW > 0
assert all(0 < lf <= 1 for lf in LOAD_FRACTIONS_TO_TEST)
assert WIND_CAPACITY_STEP_KW > 0 and BATTERY_CAPACITY_STEP_KWH > 0

GRID_LOSS_FACTOR = 1.0 + MARKET_LOSS_SHARE if APPLY_GRID_LOSSES else 1.0
RUN_START_TS = pd.Timestamp(RUN_START)
RUN_END_TS = pd.Timestamp(RUN_END)
if RUN_END_TS < RUN_START_TS:
    raise ValueError("RUN_END must be after RUN_START")

print("Imports and validation complete.")
print("Grid loss factor:", GRID_LOSS_FACTOR)


Imports and validation complete.
Grid loss factor: 1.0316426921769999



## 3. Brightway setup and electrolyser foreground decomposition

The optimisation needs two values from the existing foreground activity:

1. `direct_electricity_kwh_per_kg_h2` — electricity demand embedded in the operation process.
2. `fixed_non_electricity_kgco2e_per_kg_h2` — everything else in the foreground process, including water, electrolyte/heat if present, and amortised electrolyser capital already linked in your foreground database.

This means the notebook does **not** rebuild the foreground database for every wind/battery candidate.


In [9]:

def import_brightway():
    try:
        import bw2data as bd
        import bw2calc as bc
    except Exception as exc:
        raise ImportError(
            "Could not import Brightway. Run this notebook in the same environment/kernel "
            "where your foreground database works."
        ) from exc
    return bd, bc

bd, bc = import_brightway()
bd.projects.set_current(PROJECT_NAME)
print("Active Brightway project:", bd.projects.current)
print("Available databases:", list(bd.databases))

if FOREGROUND_DB not in bd.databases:
    raise KeyError(
        f"Foreground database {FOREGROUND_DB!r} not found in project {PROJECT_NAME!r}. "
        "Run your foreground notebook first, then rerun this notebook."
    )


def pick_lca_method():
    if LCA_METHOD is not None:
        return tuple(LCA_METHOD)
    methods = list(bd.methods)
    terms = [t.lower() for t in LCA_METHOD_SEARCH_TERMS]
    matches = []
    for method in methods:
        text = " | ".join(map(str, method)).lower()
        if all(term in text for term in terms):
            matches.append(method)
    if not matches:
        print("No method matched the search terms:", LCA_METHOD_SEARCH_TERMS)
        print("First 30 available methods:")
        for m in methods[:30]:
            print("  ", m)
        raise ValueError("Set LCA_METHOD manually in the dashboard cell.")
    print(f"Matched {len(matches)} LCIA method(s). Using index {LCA_METHOD_INDEX_IF_MULTIPLE}:")
    for i, m in enumerate(matches[:10]):
        prefix = "*" if i == LCA_METHOD_INDEX_IF_MULTIPLE else " "
        print(prefix, i, m)
    return matches[LCA_METHOD_INDEX_IF_MULTIPLE]

method = pick_lca_method()
print("Selected method:", method)


def run_lca_score(activity, method, amount=1.0):
    lca = bc.LCA({activity: amount}, method)
    lca.lci()
    lca.lcia()
    return float(lca.score)


def get_foreground_activity():
    code = ELECTROLYSER_ACTIVITY_CODES[ELECTROLYSER_TECH]
    try:
        return bd.get_activity((FOREGROUND_DB, code))
    except Exception:
        # Fallback: search by name if the code changed.
        db = bd.Database(FOREGROUND_DB)
        target = ELECTROLYSER_TECH.lower().replace(" operation", "")
        candidates = []
        for act in db:
            name = str(act.get("name", "")).lower()
            ref = str(act.get("reference product", "")).lower()
            if target in name and ("hydrogen" in name or "hydrogen" in ref):
                candidates.append(act)
        if not candidates:
            raise
        print("Could not fetch by code; using first name-search candidate:", candidates[0])
        return candidates[0]


def is_direct_electricity_exchange(exc):
    try:
        if exc.get("type") != "technosphere":
            return False
        amount = float(exc.get("amount", 0.0))
        if amount <= 0:
            return False
        inp = exc.input
        name = str(inp.get("name", "")).lower()
        unit = str(inp.get("unit", "")).lower()
        ref = str(inp.get("reference product", "")).lower()
        return ("electricity" in name or "electricity" in ref or "kilowatt hour" in unit or unit == "kwh")
    except Exception:
        return False


def decompose_foreground_electrolyser(activity, method):
    total_score = run_lca_score(activity, method, amount=1.0)
    elec_rows = []
    electricity_score_in_original = 0.0
    direct_electricity_kwh = 0.0

    for exc in activity.technosphere():
        if not is_direct_electricity_exchange(exc):
            continue
        inp = exc.input
        amount = float(exc.get("amount"))
        score_per_kwh = run_lca_score(inp, method, amount=1.0)
        electricity_score_in_original += amount * score_per_kwh
        direct_electricity_kwh += amount
        elec_rows.append({
            "input_database": inp.get("database"),
            "input_code": inp.get("code"),
            "input_name": inp.get("name"),
            "input_location": inp.get("location"),
            "input_unit": inp.get("unit"),
            "amount_kwh_per_kg_h2": amount,
            "original_score_kgco2e_per_kwh": score_per_kwh,
            "contribution_kgco2e_per_kg_h2": amount * score_per_kwh,
        })

    if direct_electricity_kwh <= 0:
        raise ValueError(
            "No direct electricity technosphere exchange found in the selected foreground activity. "
            "Check the electricity exchange name/unit in your foreground database."
        )

    fixed_non_electricity = total_score - electricity_score_in_original
    return {
        "activity": activity,
        "total_original_kgco2e_per_kg_h2": total_score,
        "direct_electricity_kwh_per_kg_h2": direct_electricity_kwh,
        "original_electricity_kgco2e_per_kg_h2": electricity_score_in_original,
        "fixed_non_electricity_kgco2e_per_kg_h2": fixed_non_electricity,
        "electricity_exchanges": pd.DataFrame(elec_rows),
    }

fg_activity = get_foreground_activity()
print("Selected foreground activity:", fg_activity)

decomp = decompose_foreground_electrolyser(fg_activity, method)
print("\nForeground decomposition:")
for k, v in decomp.items():
    if k not in {"activity", "electricity_exchanges"}:
        print(f"  {k}: {v:.6g}")

decomp["electricity_exchanges"]


Active Brightway project: hydrogen-smr
Available databases: ['ecoinvent-3.9.1-biosphere', 'ecoinvent-3.9.1-apos', 'hydrogen foreground']
Matched 16 LCIA method(s). Using index 0:
* 0 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')
  1 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change: biogenic no LT', 'global warming potential (GWP100) no LT')
  2 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change: biogenic, including SLCFs no LT', 'global warming potential (GWP100) no LT')
  3 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change: fossil no LT', 'global warming potential (GWP100) no LT')
  4 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change: fossil, including SLCFs no LT', 'global warming potential (GWP100) no LT')
  5 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change: including SLCFs no LT', 'global warming potential (GWP100) no LT')
  6 ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change: land use no LT

,input_database,input_code,input_name,input_location,input_unit,amount_kwh_per_kg_h2,original_score_kgco2e_per_kwh,contribution_kgco2e_per_kg_h2
0,ecoinvent-3.9.1-apos,199beb293c1a82f20589073ed64dcb82,"market for electricity, low voltage",DE,kilowatt hour,51.8,0.461677,23.914889


## 4. Wind electricity LCA score from Brightway

In [10]:

def activity_text(act):
    fields = [act.get("name"), act.get("reference product"), act.get("location"), act.get("unit"), act.get("database")]
    return " | ".join(str(x) for x in fields if x is not None).lower()


def search_activities(query, databases=None, limit=25):
    tokens = [t for t in query.lower().replace(",", " ").split() if t]
    if databases is None:
        databases = [db for db in bd.databases if db != FOREGROUND_DB]
    candidates = []
    for db_name in databases:
        try:
            db = bd.Database(db_name)
            for act in db:
                txt = activity_text(act)
                if all(tok in txt for tok in tokens):
                    candidates.append(act)
                    if len(candidates) >= limit:
                        return candidates
        except Exception as exc:
            warnings.warn(f"Skipping database {db_name!r}: {exc}")
    return candidates

if WIND_LCA_SCORE_OVERRIDE_KGCO2E_PER_KWH is not None:
    wind_lca_score_kgco2e_per_kwh = float(WIND_LCA_SCORE_OVERRIDE_KGCO2E_PER_KWH)
    wind_background_activity = None
    print("Using override wind LCA score:", wind_lca_score_kgco2e_per_kwh, "kgCO2e/kWh")
else:
    wind_candidates = search_activities(
        WIND_BACKGROUND_QUERY,
        databases=BACKGROUND_DATABASES_TO_SEARCH,
        limit=max(25, WIND_BACKGROUND_INDEX + 1),
    )
    if not wind_candidates:
        raise ValueError(
            "No Brightway wind background candidates found. Either adjust WIND_BACKGROUND_QUERY "
            "or set WIND_LCA_SCORE_OVERRIDE_KGCO2E_PER_KWH in the dashboard."
        )
    print("Wind background candidates:")
    for i, act in enumerate(wind_candidates[:20]):
        print(f"[{i}] {act}")
    wind_background_activity = wind_candidates[WIND_BACKGROUND_INDEX]
    wind_lca_score_kgco2e_per_kwh = run_lca_score(wind_background_activity, method, amount=1.0)
    print("\nSelected wind background activity:", wind_background_activity)
    print(f"Wind LCA score: {wind_lca_score_kgco2e_per_kwh:.6g} kgCO2e/kWh")


Wind background candidates:
[0] 'electricity production, wind, 1-3MW turbine, onshore' (kilowatt hour, GB, None)

Selected wind background activity: 'electricity production, wind, 1-3MW turbine, onshore' (kilowatt hour, GB, None)
Wind LCA score: 0.0144658 kgCO2e/kWh



## 5. Load grid CSV and build grid-emission time series

The notebook first uses `CARBON_INTENSITY` from your CSV. If that column is not available, it falls back to a generation-weighted estimate using `FUEL_LCA_FACTORS_KGCO2E_PER_KWH` from the dashboard.


In [11]:

def load_grid_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"CSV not found: {path.resolve()}\n"
            "Put df_fuel_ckan.csv in the same folder as this notebook, or edit CSV_PATH in the dashboard."
        )
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    if CSV_DATETIME_COL not in df.columns:
        datetime_candidates = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
        if not datetime_candidates:
            raise ValueError(f"Could not find datetime column. Columns: {list(df.columns)}")
        dt_col = datetime_candidates[0]
        print(f"Using detected datetime column: {dt_col}")
    else:
        dt_col = CSV_DATETIME_COL
    df["DATETIME"] = pd.to_datetime(df[dt_col], errors="coerce")
    df = df.dropna(subset=["DATETIME"]).sort_values("DATETIME").reset_index(drop=True)
    df = df[(df["DATETIME"] >= RUN_START_TS) & (df["DATETIME"] <= RUN_END_TS)].copy()
    if df.empty:
        raise ValueError("No CSV rows in selected RUN_START/RUN_END range.")
    return df


def infer_timestep_hours(datetimes):
    d = pd.Series(datetimes).sort_values().diff().dropna()
    if d.empty:
        return 0.5
    return float(d.median() / pd.Timedelta(hours=1))


def build_grid_intensity_kgco2e_per_kwh(df):
    if CSV_CARBON_INTENSITY_COL in df.columns:
        s = pd.to_numeric(df[CSV_CARBON_INTENSITY_COL], errors="coerce")
        if CSV_CARBON_INTENSITY_UNIT.lower().startswith("g"):
            s = s / 1000.0
        elif not CSV_CARBON_INTENSITY_UNIT.lower().startswith("kg"):
            raise ValueError("CSV_CARBON_INTENSITY_UNIT must start with 'g' or 'kg'.")
        source = CSV_CARBON_INTENSITY_COL
    else:
        numerator = pd.Series(0.0, index=df.index)
        denominator = pd.Series(0.0, index=df.index)
        for col, factor in FUEL_LCA_FACTORS_KGCO2E_PER_KWH.items():
            if col in df.columns:
                gen = pd.to_numeric(df[col], errors="coerce").fillna(0.0).clip(lower=0.0)
                numerator += gen * float(factor)
                denominator += gen
        if denominator.max() <= 0:
            raise ValueError("Could not compute generation-weighted grid intensity; no generation columns found.")
        s = numerator / denominator.replace(0, np.nan)
        source = "generation-weighted fallback"
    s = s.astype(float).interpolate(limit_direction="both")
    s = s * GRID_LOSS_FACTOR + GRID_LIFECYCLE_UPLIFT_KGCO2E_PER_KWH
    return s, source

grid_df = load_grid_csv(CSV_PATH)
grid_df["dt_hours"] = infer_timestep_hours(grid_df["DATETIME"])
grid_df["grid_lca_kgco2e_per_kwh"] , grid_intensity_source = build_grid_intensity_kgco2e_per_kwh(grid_df)

print(f"Loaded {len(grid_df):,} grid rows from {grid_df['DATETIME'].min()} to {grid_df['DATETIME'].max()}")
print(f"Timestep: {grid_df['dt_hours'].iloc[0]:.3g} hours")
print("Grid intensity source:", grid_intensity_source)
print("Grid intensity summary, kgCO2e/kWh delivered:")
display(grid_df["grid_lca_kgco2e_per_kwh"].describe().to_frame().T)

grid_df.head()


Loaded 17,520 grid rows from 2025-01-01 00:00:00 to 2025-12-31 23:30:00
Timestep: 0.5 hours
Grid intensity source: CARBON_INTENSITY
Grid intensity summary, kgCO2e/kWh delivered:


,count,mean,std,min,25%,50%,75%,max
grid_lca_kgco2e_per_kwh,17520.0,0.130868,0.060303,0.022696,0.078405,0.123797,0.176411,0.304335


,DATETIME,GAS,COAL,NUCLEAR,WIND,WIND_EMB,HYDRO,IMPORTS,BIOMASS,OTHER,SOLAR,STORAGE,GENERATION,CARBON_INTENSITY,LOW_CARBON,ZERO_CARBON,RENEWABLE,FOSSIL,GAS_perc,COAL_perc,NUCLEAR_perc,WIND_perc,WIND_EMB_perc,HYDRO_perc,IMPORTS_perc,BIOMASS_perc,OTHER_perc,SOLAR_perc,STORAGE_perc,GENERATION_perc,LOW_CARBON_perc,ZERO_CARBON_perc,RENEWABLE_perc,FOSSIL_perc,dt_hours,grid_lca_kgco2e_per_kwh
280512,2025-01-01 00:00:00,3711.0,0.0,5062.0,15799.0,5452.0,744.0,336.0,962.0,223.0,0.0,0.0,32289.0,51.0,28019.0,22567.0,21995.0,3711.0,11.5,0.0,15.7,48.9,16.9,2.3,1.0,3.0,0.7,0.0,0.0,100.0,86.8,85.2,68.1,11.5,0.5,0.052614
280513,2025-01-01 00:30:00,3935.0,0.0,5059.0,15317.0,5410.0,745.0,318.0,1097.0,281.0,0.0,0.0,32162.0,55.0,27628.0,22232.0,21472.0,3935.0,12.2,0.0,15.7,47.6,16.8,2.3,1.0,3.4,0.9,0.0,0.0,100.0,85.9,88.8,66.8,12.2,0.5,0.056740
280514,2025-01-01 01:00:00,3769.0,0.0,5056.0,14992.0,5358.0,744.0,481.0,1106.0,293.0,0.0,0.0,31799.0,54.0,27256.0,21907.0,21094.0,3769.0,11.9,0.0,15.9,47.1,16.8,2.3,1.5,3.5,0.9,0.0,0.0,100.0,85.7,89.0,66.3,11.9,0.5,0.055709
280515,2025-01-01 01:30:00,3719.0,0.0,5057.0,14580.0,5236.0,745.0,454.0,1083.0,223.0,0.0,0.0,31097.0,54.0,26701.0,21465.0,20561.0,3719.0,12.0,0.0,16.3,46.9,16.8,2.4,1.5,3.5,0.7,0.0,0.0,100.0,85.9,89.0,66.1,12.0,0.5,0.055709
280516,2025-01-01 02:00:00,3675.0,0.0,5057.0,14685.0,5115.0,737.0,286.0,1007.0,229.0,0.0,0.0,30791.0,53.0,26601.0,21486.0,20537.0,3675.0,11.9,0.0,16.4,47.7,16.6,2.4,0.9,3.3,0.7,0.0,0.0,100.0,86.4,88.9,66.7,11.9,0.5,0.054677


## 6. Fetch Renewables.ninja wind data and align it to the grid rows

In [12]:

def get_ninja_token():
    token = (NINJA_API_TOKEN or "").strip()
    if token:
        return token
    token = os.environ.get("RENEWABLES_NINJA_TOKEN", "").strip()
    if token:
        return token
    return getpass("Renewables.ninja API token: ").strip()


def parse_ninja_index(index):
    """Renewables.ninja sometimes returns epoch-millisecond keys (e.g. '1735689600000')
    and sometimes ISO datetime strings. Handle both robustly."""
    idx_str = pd.Index(index).astype(str)
    if idx_str.str.fullmatch(r"\d+").all():
        values = idx_str.astype("int64")
        # Distinguish seconds vs milliseconds by magnitude (ms keys are ~1e12).
        unit = "ms" if values.max() >= 1_000_000_000_000 else "s"
        return pd.to_datetime(values, unit=unit)
    return pd.to_datetime(idx_str)


def fetch_renewables_ninja_wind():
    token = get_ninja_token()
    if not token:
        raise RuntimeError("No Renewables.ninja token supplied.")

    date_from = RUN_START_TS.date().isoformat()
    date_to = RUN_END_TS.date().isoformat()
    url = "https://www.renewables.ninja/api/data/wind"
    params = {
        "lat": WIND_LAT,
        "lon": WIND_LON,
        "date_from": date_from,
        "date_to": date_to,
        "dataset": NINJA_DATASET,
        "capacity": WIND_API_REFERENCE_CAPACITY_KW,
        "height": NINJA_HUB_HEIGHT_M,
        "turbine": NINJA_TURBINE,
        "format": "json",
    }
    headers = {"Authorization": f"Token {token}"}
    r = requests.get(url, params=params, headers=headers, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(
            f"Renewables.ninja request failed with status {r.status_code}.\n"
            f"Response preview: {r.text[:500]}"
        )
    payload = r.json()
    if "data" not in payload:
        raise ValueError(f"Unexpected Renewables.ninja payload keys: {payload.keys()}")
    wind = pd.DataFrame.from_dict(payload["data"], orient="index")
    wind.index = parse_ninja_index(wind.index)
    wind = wind.sort_index()
    wind.index.name = "DATETIME"
    wind = wind.reset_index()

    # Usually the column is called 'electricity'. Be defensive in case API schema changes.
    if "electricity" in wind.columns:
        power_col = "electricity"
    else:
        possible = [c for c in wind.columns if any(term in c.lower() for term in ["electric", "power", "output"])]
        if not possible:
            raise ValueError(f"Could not identify wind power column. Columns: {list(wind.columns)}")
        power_col = possible[0]
        print(f"Using detected wind power column: {power_col}")
    wind["wind_power_per_kw_installed"] = pd.to_numeric(wind[power_col], errors="coerce") / float(WIND_API_REFERENCE_CAPACITY_KW)
    wind["wind_power_per_kw_installed"] = wind["wind_power_per_kw_installed"].interpolate(limit_direction="both").clip(lower=0.0)

    if wind["wind_power_per_kw_installed"].max() > 1.25:
        warnings.warn(
            "Wind output per kW installed exceeds 1.25. Check whether Renewables.ninja returned "
            "energy rather than power, or whether the reference capacity was interpreted differently."
        )
    return wind, payload.get("metadata", {}), params


def align_wind_to_grid(wind_df, grid_df):
    w = wind_df[["DATETIME", "wind_power_per_kw_installed"]].copy().sort_values("DATETIME")
    g = grid_df[["DATETIME"]].copy().sort_values("DATETIME")
    if WIND_TO_GRID_ALIGNMENT == "ffill":
        aligned = pd.merge_asof(g, w, on="DATETIME", direction="backward")
        aligned["wind_power_per_kw_installed"] = aligned["wind_power_per_kw_installed"].interpolate(limit_direction="both")
    else:
        w2 = w.set_index("DATETIME").sort_index()
        target_index = pd.DatetimeIndex(g["DATETIME"])
        union_index = w2.index.union(target_index)
        interp = w2.reindex(union_index).interpolate(method="time").reindex(target_index)
        aligned = pd.DataFrame({
            "DATETIME": target_index,
            "wind_power_per_kw_installed": interp["wind_power_per_kw_installed"].to_numpy(),
        })
    return aligned

wind_hourly_df, ninja_metadata, ninja_params = fetch_renewables_ninja_wind()
wind_aligned = align_wind_to_grid(wind_hourly_df, grid_df)

model_df = grid_df.merge(wind_aligned, on="DATETIME", how="left")
model_df["wind_power_per_kw_installed"] = model_df["wind_power_per_kw_installed"].interpolate(limit_direction="both").clip(lower=0.0)

print(f"Fetched {len(wind_hourly_df):,} wind rows from Renewables.ninja.")
print("API params used:", ninja_params)
print("Aligned model rows:", len(model_df))
print("Wind capacity-factor-like summary per kW installed:")
display(model_df["wind_power_per_kw_installed"].describe().to_frame().T)
model_df.head()


Fetched 8,760 wind rows from Renewables.ninja.
API params used: {'lat': 51.732, 'lon': -0.3711, 'date_from': '2025-01-01', 'date_to': '2025-12-31', 'dataset': 'merra2', 'capacity': 1.0, 'height': 80, 'turbine': 'Vestas V90 2000', 'format': 'json'}
Aligned model rows: 17520
Wind capacity-factor-like summary per kW installed:


,count,mean,std,min,25%,50%,75%,max
wind_power_per_kw_installed,17520.0,0.349075,0.228463,0.001,0.172,0.316,0.476,0.991


,DATETIME,GAS,COAL,NUCLEAR,WIND,WIND_EMB,HYDRO,IMPORTS,BIOMASS,OTHER,SOLAR,STORAGE,GENERATION,CARBON_INTENSITY,LOW_CARBON,ZERO_CARBON,RENEWABLE,FOSSIL,GAS_perc,COAL_perc,NUCLEAR_perc,WIND_perc,WIND_EMB_perc,HYDRO_perc,IMPORTS_perc,BIOMASS_perc,OTHER_perc,SOLAR_perc,STORAGE_perc,GENERATION_perc,LOW_CARBON_perc,ZERO_CARBON_perc,RENEWABLE_perc,FOSSIL_perc,dt_hours,grid_lca_kgco2e_per_kwh,wind_power_per_kw_installed
0,2025-01-01 00:00:00,3711.0,0.0,5062.0,15799.0,5452.0,744.0,336.0,962.0,223.0,0.0,0.0,32289.0,51.0,28019.0,22567.0,21995.0,3711.0,11.5,0.0,15.7,48.9,16.9,2.3,1.0,3.0,0.7,0.0,0.0,100.0,86.8,85.2,68.1,11.5,0.5,0.052614,0.976
1,2025-01-01 00:30:00,3935.0,0.0,5059.0,15317.0,5410.0,745.0,318.0,1097.0,281.0,0.0,0.0,32162.0,55.0,27628.0,22232.0,21472.0,3935.0,12.2,0.0,15.7,47.6,16.8,2.3,1.0,3.4,0.9,0.0,0.0,100.0,85.9,88.8,66.8,12.2,0.5,0.056740,0.976
2,2025-01-01 01:00:00,3769.0,0.0,5056.0,14992.0,5358.0,744.0,481.0,1106.0,293.0,0.0,0.0,31799.0,54.0,27256.0,21907.0,21094.0,3769.0,11.9,0.0,15.9,47.1,16.8,2.3,1.5,3.5,0.9,0.0,0.0,100.0,85.7,89.0,66.3,11.9,0.5,0.055709,0.976
3,2025-01-01 01:30:00,3719.0,0.0,5057.0,14580.0,5236.0,745.0,454.0,1083.0,223.0,0.0,0.0,31097.0,54.0,26701.0,21465.0,20561.0,3719.0,12.0,0.0,16.3,46.9,16.8,2.4,1.5,3.5,0.7,0.0,0.0,100.0,85.9,89.0,66.1,12.0,0.5,0.055709,0.976
4,2025-01-01 02:00:00,3675.0,0.0,5057.0,14685.0,5115.0,737.0,286.0,1007.0,229.0,0.0,0.0,30791.0,53.0,26601.0,21486.0,20537.0,3675.0,11.9,0.0,16.4,47.7,16.6,2.4,0.9,3.3,0.7,0.0,0.0,100.0,86.4,88.9,66.7,11.9,0.5,0.054677,0.977



## 7. Dispatch model and LCA calculation

For each candidate:

1. Wind generation is scaled from the 1 kW Renewables.ninja profile.
2. The electrolyser runs at a fixed load fraction: 100% or 10% by default.
3. Surplus wind charges the battery.
4. If wind is short, the battery discharges.
5. Any remaining deficit is supplied by the grid.
6. The candidate passes if its GWP result meets the selected threshold rule.


In [14]:
def simulate_candidate(df, wind_capacity_kw, battery_capacity_kwh, load_fraction, return_timeseries=False):
    """
    Simulate a wind/battery/electrolyser candidate over the time series.
    
    Parameters:
    -----------
    df : DataFrame
        model_df with columns DATETIME, dt_hours, grid_lca_kgco2e_per_kwh, wind_power_per_kw_installed
    wind_capacity_kw : float
        Wind farm capacity in kW
    battery_capacity_kwh : float
        Battery energy capacity in kWh
    load_fraction : float
        Electrolyser load fraction (0-1)
    return_timeseries : bool
        If True, also return the full time series
    
    Returns:
    --------
    dict if return_timeseries=False, else (dict, DataFrame)
    """
    ts = df.copy()
    ts["demand_kwh"] = ELECTROLYSER_CAPACITY_KW * load_fraction * ts["dt_hours"]
    ts["wind_generation_kwh"] = wind_capacity_kw * ts["wind_power_per_kw_installed"] * ts["dt_hours"]
    ts["soc_kwh"] = 0.0
    ts["battery_charge_kwh"] = 0.0
    ts["battery_discharge_kwh"] = 0.0
    ts["battery_discharge_to_load_kwh"] = 0.0
    ts["grid_kwh"] = 0.0
    
    soc = battery_capacity_kwh * BATTERY_INITIAL_SOC_FRACTION
    
    for idx in ts.index:
        demand = ts.loc[idx, "demand_kwh"]
        wind_gen = ts.loc[idx, "wind_generation_kwh"]
        max_charge = min(BATTERY_C_RATE * battery_capacity_kwh * ts.loc[idx, "dt_hours"],
                         (battery_capacity_kwh - soc))
        max_discharge = min(BATTERY_C_RATE * battery_capacity_kwh * ts.loc[idx, "dt_hours"], soc)
        
        # First priority: use wind directly or to charge battery
        available_for_load = wind_gen
        available_for_battery_charge = 0.0
        
        if available_for_load >= demand:
            # Wind covers demand
            load_from_wind = demand
            available_for_battery_charge = (wind_gen - demand)
        else:
            # Wind partially covers demand
            load_from_wind = available_for_load
            deficit = demand - load_from_wind
            
            # Try to cover deficit from battery
            discharge = min(max_discharge, deficit)
            soc -= discharge / BATTERY_ROUNDTRIP_EFFICIENCY
            ts.loc[idx, "battery_discharge_kwh"] = discharge
            ts.loc[idx, "battery_discharge_to_load_kwh"] = discharge
            
            # Remaining deficit from grid
            grid_needed = deficit - discharge
            ts.loc[idx, "grid_kwh"] = grid_needed
        
        ts.loc[idx, "battery_charge_kwh"] = min(available_for_battery_charge, max_charge)
        soc += ts.loc[idx, "battery_charge_kwh"] * BATTERY_ROUNDTRIP_EFFICIENCY
        soc = np.clip(soc, 0, battery_capacity_kwh)
        ts.loc[idx, "soc_kwh"] = soc
    
    # LCA calculation
    direct_electricity_kwh_per_kg_h2 = decomp["direct_electricity_kwh_per_kg_h2"]
    fixed_non_electricity_kgco2e_per_kg_h2 = decomp["fixed_non_electricity_kgco2e_per_kg_h2"]
    
    ts["h2_kg"] = ts["demand_kwh"] / direct_electricity_kwh_per_kg_h2
    
    # Electricity sources for H2 production
    total_h2_kwh_demand = ts["demand_kwh"].sum()
    if total_h2_kwh_demand > 0:
        wind_share = ts["wind_generation_kwh"].sum() / total_h2_kwh_demand
        grid_share = ts["grid_kwh"].sum() / total_h2_kwh_demand
    else:
        wind_share = grid_share = 0.0
    
    # GWP per slice
    ts["grid_lca_kgco2e_per_kg_h2"] = (
        ts["grid_kwh"] / ts["h2_kg"].replace(0, np.nan) * ts["grid_lca_kgco2e_per_kwh"]
    )
    ts["wind_lca_kgco2e_per_kg_h2"] = (
        ts["wind_generation_kwh"] / ts["h2_kg"].replace(0, np.nan) * wind_lca_score_kgco2e_per_kwh
    )
    ts["battery_lca_kgco2e_per_kg_h2"] = (
        ts["battery_discharge_to_load_kwh"] / ts["h2_kg"].replace(0, np.nan) 
        * BATTERY_EMBODIED_KGCO2E_PER_KWH_CAPACITY / BATTERY_LIFETIME_YEARS * 365.25 / 24
        if INCLUDE_BATTERY_CAPITAL_IN_LCA else 0.0
    )
    ts["slice_gwp_kgco2e_per_kg_h2"] = (
        ts["grid_lca_kgco2e_per_kg_h2"]
        + ts["wind_lca_kgco2e_per_kg_h2"]
        + ts["battery_lca_kgco2e_per_kg_h2"]
        + fixed_non_electricity_kgco2e_per_kg_h2
    )
    
    # Annual/period average
    total_h2_kg = ts["h2_kg"].sum()
    if total_h2_kg <= 0:
        avg_gwp = np.nan
    else:
        total_gwp_kg = (
            ts["grid_lca_kgco2e_per_kg_h2"].sum()
            + ts["wind_lca_kgco2e_per_kg_h2"].sum()
            + ts["battery_lca_kgco2e_per_kg_h2"].sum()
            + fixed_non_electricity_kgco2e_per_kg_h2 * total_h2_kg
        )
        avg_gwp = total_gwp_kg / total_h2_kg
    
    # Compliance check
    slice_gwp = ts["slice_gwp_kgco2e_per_kg_h2"].dropna().values
    below = slice_gwp <= THRESHOLD_KGCO2E_PER_KGH2
    slice_share_below = np.mean(below) if len(below) > 0 else 0.0
    
    if COMPLIANCE_MODE == "annual_average":
        feasible = avg_gwp <= THRESHOLD_KGCO2E_PER_KGH2
    elif COMPLIANCE_MODE == "all_slices":
        valid = ~np.isnan(slice_gwp)
        feasible = bool(below[valid].all()) if valid.any() else False
    else:
        feasible = slice_share_below >= REQUIRED_SLICE_SHARE
    
    # Curtailment
    total_wind_generated = ts["wind_generation_kwh"].sum()
    total_wind_used = ts["wind_generation_kwh"].sum() - ts["grid_kwh"].sum()
    curtailment_share = (total_wind_generated - total_wind_used) / total_wind_generated if total_wind_generated > 0 else 0.0
    
    result = {
        "load_fraction": load_fraction,
        "electrolyser_power_kw": ELECTROLYSER_CAPACITY_KW * load_fraction,
        "wind_capacity_kw": wind_capacity_kw,
        "battery_capacity_kwh": battery_capacity_kwh,
        "meets_threshold": feasible,
        "avg_gwp_kgco2e_per_kg_h2": avg_gwp,
        "max_slice_gwp_kgco2e_per_kg_h2": np.nanmax(slice_gwp) if len(slice_gwp) > 0 else np.nan,
        "slice_share_below_threshold": slice_share_below,
        "total_h2_kg": total_h2_kg,
        "grid_share_of_electrolyser_demand": grid_share,
        "wind_share_of_electrolyser_demand": wind_share,
        "curtailment_share_of_wind_generation": curtailment_share,
    }
    
    if return_timeseries:
        return result, ts
    return result

# Extract decomposition values into module scope for use in simulate_candidate
direct_electricity_kwh_per_kg_h2 = decomp["direct_electricity_kwh_per_kg_h2"]
fixed_non_electricity_kgco2e_per_kg_h2 = decomp["fixed_non_electricity_kgco2e_per_kg_h2"]


## 8. Run the optimisation grid search

In [ ]:

def make_grid(start, stop, step):
    values = np.arange(float(start), float(stop) + step * 0.5, float(step))
    # Round to avoid ugly binary floats in outputs.
    decimals = max(0, int(abs(math.floor(math.log10(step)))) + 2) if step < 1 else 3
    return np.round(values, decimals)

wind_capacity_grid_kw = make_grid(WIND_CAPACITY_MIN_KW, WIND_CAPACITY_MAX_KW, WIND_CAPACITY_STEP_KW)
battery_capacity_grid_kwh = make_grid(BATTERY_CAPACITY_MIN_KWH, BATTERY_CAPACITY_MAX_KWH, BATTERY_CAPACITY_STEP_KWH)

n_candidates = len(LOAD_FRACTIONS_TO_TEST) * len(wind_capacity_grid_kw) * len(battery_capacity_grid_kwh)
print(f"Testing {n_candidates:,} candidates:")
print(f"  load fractions: {LOAD_FRACTIONS_TO_TEST}")
print(f"  wind capacities: {len(wind_capacity_grid_kw):,} values from {wind_capacity_grid_kw.min()} to {wind_capacity_grid_kw.max()} kW")
print(f"  battery capacities: {len(battery_capacity_grid_kwh):,} values from {battery_capacity_grid_kwh.min()} to {battery_capacity_grid_kwh.max()} kWh")

records = []
t0 = time.time()
count = 0
for load_fraction in LOAD_FRACTIONS_TO_TEST:
    for wind_kw in wind_capacity_grid_kw:
        for batt_kwh in battery_capacity_grid_kwh:
            count += 1
            records.append(simulate_candidate(model_df, wind_kw, batt_kwh, load_fraction))
            if PROGRESS_EVERY_N_CANDIDATES and (count % PROGRESS_EVERY_N_CANDIDATES == 0 or count == n_candidates):
                elapsed = time.time() - t0
                print(f"  {count:,}/{n_candidates:,} candidates tested ({elapsed:.1f} s)")

results_df = pd.DataFrame(records)
print("Done.")
print("Feasible candidates by load fraction:")
display(results_df.groupby("load_fraction")["meets_threshold"].agg(["sum", "count"]))
results_df.head()


Testing 201,402 candidates:
  load fractions: [1.0, 0.1]
  wind capacities: 201 values from 0.0 to 20.0 kW
  battery capacities: 501 values from 0.0 to 50.0 kWh


## 9. Select optimum and Pareto frontier

In [ ]:

def sort_feasible(df):
    if OPTIMISATION_OBJECTIVE == "min_wind_then_battery":
        return df.sort_values(["wind_capacity_kw", "battery_capacity_kwh", "avg_gwp_kgco2e_per_kg_h2"])
    if OPTIMISATION_OBJECTIVE == "min_battery_then_wind":
        return df.sort_values(["battery_capacity_kwh", "wind_capacity_kw", "avg_gwp_kgco2e_per_kg_h2"])
    out = df.copy()
    out["total_capacity_proxy"] = out["wind_capacity_kw"] + out["battery_capacity_kwh"]
    return out.sort_values(["total_capacity_proxy", "wind_capacity_kw", "battery_capacity_kwh", "avg_gwp_kgco2e_per_kg_h2"])


def pareto_frontier_2d(df):
    # Non-dominated on wind capacity and battery capacity among feasible rows.
    if df.empty:
        return df.copy()
    x = df["wind_capacity_kw"].to_numpy()
    y = df["battery_capacity_kwh"].to_numpy()
    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        dominated = (x <= x[i]) & (y <= y[i]) & ((x < x[i]) | (y < y[i]))
        if dominated.any():
            keep[i] = False
    return df.loc[keep].sort_values(["wind_capacity_kw", "battery_capacity_kwh"])

selected_rows = []
pareto_rows = []
for load_fraction, group in results_df.groupby("load_fraction", sort=False):
    feasible = group[group["meets_threshold"]].copy()
    if feasible.empty:
        print(f"No feasible candidate for load_fraction={load_fraction} within the search bounds.")
        continue
    selected_rows.append(sort_feasible(feasible).iloc[0])
    pareto_rows.append(pareto_frontier_2d(feasible))

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True) if selected_rows else pd.DataFrame()
pareto_df = pd.concat(pareto_rows, ignore_index=True) if pareto_rows else pd.DataFrame()

print("Selected optimum by load fraction:")
display(selected_df[[
    "load_fraction", "electrolyser_power_kw", "wind_capacity_kw", "battery_capacity_kwh",
    "avg_gwp_kgco2e_per_kg_h2", "max_slice_gwp_kgco2e_per_kg_h2", "slice_share_below_threshold",
    "total_h2_kg", "grid_share_of_electrolyser_demand", "wind_share_of_electrolyser_demand",
    "curtailment_share_of_wind_generation"
]] if not selected_df.empty else selected_df)

print("Pareto frontier preview:")
display(pareto_df[[
    "load_fraction", "wind_capacity_kw", "battery_capacity_kwh", "avg_gwp_kgco2e_per_kg_h2",
    "grid_share_of_electrolyser_demand", "curtailment_share_of_wind_generation"
]].head(30) if not pareto_df.empty else pareto_df)


## 10. Plot feasible region and selected dispatch

In [ ]:

# Feasible-region scatter: one figure per operating mode.
for load_fraction, group in results_df.groupby("load_fraction", sort=False):
    feasible = group[group["meets_threshold"]]
    plt.figure(figsize=(8, 5))
    plt.scatter(group["wind_capacity_kw"], group["battery_capacity_kwh"], s=8, alpha=0.15, label="tested")
    if not feasible.empty:
        plt.scatter(feasible["wind_capacity_kw"], feasible["battery_capacity_kwh"], s=10, alpha=0.7, label="feasible")
        if not selected_df.empty and load_fraction in set(selected_df["load_fraction"]):
            sel = selected_df[selected_df["load_fraction"] == load_fraction].iloc[0]
            plt.scatter([sel["wind_capacity_kw"]], [sel["battery_capacity_kwh"]], s=90, marker="x", label="selected")
    plt.xlabel("Wind capacity (kW)")
    plt.ylabel("Battery capacity (kWh)")
    plt.title(f"Feasible region — load fraction {load_fraction:g}")
    plt.legend()
    plt.show()

# Re-run selected dispatch for the first selected solution and plot a manageable preview.
if not selected_df.empty:
    selected_solution = selected_df.iloc[0]
    selected_summary, selected_ts = simulate_candidate(
        model_df,
        wind_capacity_kw=selected_solution["wind_capacity_kw"],
        battery_capacity_kwh=selected_solution["battery_capacity_kwh"],
        load_fraction=selected_solution["load_fraction"],
        return_timeseries=True,
    )
    print("Selected dispatch summary:")
    display(pd.Series(selected_summary).to_frame("value"))

    preview_days = 14
    preview_rows = int(round(preview_days * 24 / selected_ts["dt_hours"].median()))
    preview = selected_ts.head(preview_rows).copy()
    plot_cols = ["demand_kwh", "wind_generation_kwh", "grid_kwh", "battery_discharge_to_load_kwh", "soc_kwh"]
    preview.set_index("DATETIME")[plot_cols].plot(figsize=(12, 5))
    plt.ylabel("kWh per time slice, or kWh stored for SOC")
    plt.title(f"Selected dispatch preview — first {preview_days} days")
    plt.show()

    selected_ts.set_index("DATETIME")[["slice_gwp_kgco2e_per_kg_h2", "grid_lca_kgco2e_per_kwh"]].plot(figsize=(12, 5))
    plt.ylabel("kgCO2e/kgH2, or kgCO2e/kWh")
    plt.title("Selected solution — slice GWP and grid intensity")
    plt.show()
else:
    selected_ts = pd.DataFrame()
    print("No selected solution to plot. Increase search bounds or relax the threshold/compliance mode.")


## 11. Save outputs

In [ ]:

if SAVE_OUTPUTS:
    outdir = Path(OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    safe_turbine = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in NINJA_TURBINE)[:60]
    prefix = f"opt_{ELECTROLYSER_TECH.replace(' ', '_')}_{safe_turbine}_{NINJA_HUB_HEIGHT_M}m_{stamp}"

    results_path = outdir / f"{prefix}_all_candidates.csv"
    selected_path = outdir / f"{prefix}_selected.csv"
    pareto_path = outdir / f"{prefix}_pareto.csv"
    dispatch_path = outdir / f"{prefix}_selected_dispatch.csv"
    dashboard_path = outdir / f"{prefix}_dashboard.json"

    results_df.to_csv(results_path, index=False)
    selected_df.to_csv(selected_path, index=False)
    pareto_df.to_csv(pareto_path, index=False)
    if "selected_ts" in globals() and not selected_ts.empty:
        selected_ts.to_csv(dispatch_path, index=False)

    dashboard_snapshot = {
        "PROJECT_NAME": PROJECT_NAME,
        "FOREGROUND_DB": FOREGROUND_DB,
        "ELECTROLYSER_TECH": ELECTROLYSER_TECH,
        "ELECTROLYSER_CAPACITY_KW": ELECTROLYSER_CAPACITY_KW,
        "LOAD_FRACTIONS_TO_TEST": LOAD_FRACTIONS_TO_TEST,
        "CSV_PATH": CSV_PATH,
        "RUN_START": RUN_START,
        "RUN_END": RUN_END,
        "WIND_LAT": WIND_LAT,
        "WIND_LON": WIND_LON,
        "NINJA_DATASET": NINJA_DATASET,
        "NINJA_TURBINE": NINJA_TURBINE,
        "NINJA_HUB_HEIGHT_M": NINJA_HUB_HEIGHT_M,
        "WIND_ALLOCATION_MODE": WIND_ALLOCATION_MODE,
        "COMPLIANCE_MODE": COMPLIANCE_MODE,
        "THRESHOLD_KGCO2E_PER_KGH2": THRESHOLD_KGCO2E_PER_KGH2,
        "WIND_CAPACITY_MIN_KW": WIND_CAPACITY_MIN_KW,
        "WIND_CAPACITY_MAX_KW": WIND_CAPACITY_MAX_KW,
        "WIND_CAPACITY_STEP_KW": WIND_CAPACITY_STEP_KW,
        "BATTERY_CAPACITY_MIN_KWH": BATTERY_CAPACITY_MIN_KWH,
        "BATTERY_CAPACITY_MAX_KWH": BATTERY_CAPACITY_MAX_KWH,
        "BATTERY_CAPACITY_STEP_KWH": BATTERY_CAPACITY_STEP_KWH,
        "direct_electricity_kwh_per_kg_h2": direct_electricity_kwh_per_kg_h2,
        "fixed_non_electricity_kgco2e_per_kg_h2": fixed_non_electricity_kgco2e_per_kg_h2,
        "wind_lca_score_kgco2e_per_kwh": wind_lca_score_kgco2e_per_kwh,
        "method": list(method),
    }
    with open(dashboard_path, "w", encoding="utf-8") as f:
        json.dump(dashboard_snapshot, f, indent=2)

    print("Saved:")
    print(" ", results_path.resolve())
    print(" ", selected_path.resolve())
    print(" ", pareto_path.resolve())
    if "selected_ts" in globals() and not selected_ts.empty:
        print(" ", dispatch_path.resolve())
    print(" ", dashboard_path.resolve())
else:
    print("SAVE_OUTPUTS is False; nothing saved.")



## Notes on interpretation

- The wind API is called once for the selected site, turbine model and hub height, then scaled by candidate wind-farm capacity. This is appropriate because capacity is a linear scaling after the turbine/power-curve profile has been generated.
- The battery is a simple wind-charged buffer, not a full MILP dispatch model. It does not charge from grid electricity.
- `WIND_ALLOCATION_MODE = "dedicated_no_export"` is conservative for a dedicated wind farm because curtailed wind still carries wind-infrastructure burden. Use `"used_only"` if you treat surplus wind as exported or outside the hydrogen system boundary.
- If no feasible solution appears, increase `WIND_CAPACITY_MAX_KW` and/or `BATTERY_CAPACITY_MAX_KWH`, relax `COMPLIANCE_MODE`, or check whether the foreground activity already has a fixed non-electricity burden above the threshold.
